# Lab 4: AI Weather Forecasting with FourCastNet

Generate a 48-hour weather forecast using pretrained FourCastNet, compare it with ERA5 reanalysis, and examine how forecast errors change with lead time.

Start from the atmosphere at **00:00 UTC on 1 September 2022**. The model predicts eight six-hour steps without seeing the future ERA5 fields.

By the end of this lab, you should be able to:

- Explain how a neural operator maps a global weather state to the next state.
- Compare a forecast with reanalysis and a simple persistence baseline.
- Identify forecast errors that an appealing animation can hide.

This is an **inference and verification** lab, not a training exercise. ERA5 combines observations and a numerical model; it is reanalysis, not direct observation.

[Start Here](../../Start_Here.ipynb) · Previous: [Lab 3: Heat Conduction](../03_heat_conduction/Lab_3_Heat_Conduction.ipynb) · Next: [Challenge 1: Wave Dynamics](../../02_challenges/01_wave/Challenge_1_Wave_Dynamics.ipynb)

## From one weather state to the next

FourCastNet uses an **Adaptive Fourier Neural Operator (AFNO)**. Its layers mix spatial information in Fourier space. We use NVIDIA's pretrained **26-channel checkpoint**, which receives surface and pressure-level weather fields on a global **720 × 1440** grid.

![Global 26-field weather grid and eight six-hour forecast steps; East Asia is a verification crop, not the model domain.](../../ETC/assets/teaching/lab4-domain.svg)

**Find the grid in code:** [run_forecast.py](source_code/run_forecast.py), `LATITUDE`, `LONGITUDE` and `load_initial()`. The East Asia crop is defined by `EAST_ASIA` and `region_indices()` in [evaluate_weather.py](source_code/evaluate_weather.py).

### Follow the code: inference, then verification

![Code flow from initial ERA5 data through the fixed PhysicsNeMo AFNO model, recurrent forecast and separate ERA5 verification.](../../ETC/assets/teaching/lab4-workflow.svg)

**Trace it in code:** [run_forecast.py](source_code/run_forecast.py), `load_model()` → `main()` → `torch.inference_mode()`; then [evaluate_weather.py](source_code/evaluate_weather.py), `load_pair()` → `evaluate()`. The notebook's `run_script()` starts these programs.

The output is the complete next state, not an increment. Each prediction becomes the next input, so errors can propagate. Future ERA5 fields enter only verification. Unlike Labs 1–3, this lab does not update model weights or use a PINN loss.

## 1. Prepare the model and data

Use the **AI4Sci PhysicsNeMo** kernel. The default device is CUDA when available. The setup cell reports the selected device; CPU execution is possible but slower.

The first preparation downloads the checkpoint (about 301 MB) and this case's ERA5 source data (about 622 MB compressed). Cached files are reused. A Launchable that has prepared these files does not need to download them again. Download time is separate from forecast time.

This notebook uses the supplied course environment and does not install packages.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import time
import uuid

import numpy as np
import torch
from IPython.display import Image, Markdown, display

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'Start_Here.ipynb').is_file()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook inside the bootcamp repository.')
LAB = ROOT / '01_labs/04_weather_forecasting'
SOURCE = LAB / 'source_code'
ASSETS = LAB / 'assets'
CACHE = Path(os.environ.get('AI4SCI_WEATHER_CACHE',
                            '~/.cache/ai4sci/weather')).expanduser().resolve()
DATA = CACHE / 'data-20220901'
COASTLINE = ASSETS / 'coastline.geojson'
COASTLINE_CREDIT = 'Natural Earth 1:110m coastline (public domain)'
CASE_TIME = '2022-09-01T00:00:00Z'
DEVICE = os.environ.get('AI4SCI_DEVICE', 'auto').strip().lower()
if DEVICE == 'auto':
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE not in {'cpu', 'cuda'}:
    raise ValueError('AI4SCI_DEVICE must be auto, cuda, or cpu.')
if DEVICE == 'cuda' and not torch.cuda.is_available():
    raise RuntimeError('CUDA was requested but is unavailable. Check the GPU and selected kernel.')
sys.path.insert(0, str(SOURCE))
RUN_COMPLETED = False
EVALUATION_COMPLETED = False
print('Python:', sys.executable)
print('Device:', torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU (slower execution)')
print('Cache:', CACHE)

def run_script(script, *arguments):
    command = [sys.executable, '-u', str(SOURCE / script), *map(str, arguments)]
    started = time.perf_counter()
    with subprocess.Popen(command, cwd=LAB, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        if process.wait():
            raise subprocess.CalledProcessError(process.returncode, command)
    return time.perf_counter() - started

In [ ]:
preparation_seconds = run_script('prepare.py', '--cache-dir', CACHE)
provenance = json.loads((DATA / 'provenance.json').read_text())
assert provenance['initial_time_utc'] == CASE_TIME
print(f'Preparation and cache verification: {preparation_seconds:.1f} seconds')
print('The initial state and separate verification data are ready.')

The input retains the original 0.25° latitude–longitude grid, except for the south-pole row omitted by this checkpoint. Relative humidity is derived from ERA5 temperature and specific humidity using the documented IFS formula. `provenance.json` records the sources and preprocessing.

`initial.npz` contains only the starting state. `truth.npz` contains the ERA5 fields used for verification. It is never passed to the forecasting script.

In [ ]:
with np.load(DATA / 'initial.npz', allow_pickle=False) as saved:
    initial_state = saved['state']
    print('Input shape (variable, latitude, longitude):', initial_state.shape)
    print('Initialization:', np.datetime_as_string(saved['time'], unit='h'), 'UTC')
    print('Variables:', ', '.join(saved['variables'].tolist()))
    print('Latitude:', saved['latitude'][0], 'to', saved['latitude'][-1], 'degrees')
    print('Longitude:', saved['longitude'][0], 'to', saved['longitude'][-1], 'degrees')
del initial_state

## 2. Forecast 48 hours

Run eight six-hour steps. **48 hours is the forecast horizon, not the time you need to wait.** Progress messages report each lead and the measured computation time.

Each execution saves a separate results folder. The timer around this cell includes model loading and writing the output; the model-only timer does not.

In [ ]:
RUN_COMPLETED = False
EVALUATION_COMPLETED = False
RUN = LAB / 'outputs' / f'forecast-{uuid.uuid4().hex[:10]}'
forecast_seconds = run_script('run_forecast.py',
                              '--initial', DATA / 'initial.npz',
                              '--model-dir', CACHE / 'model',
                              '--output', RUN, '--device', DEVICE, '--steps', '8')
RUN_COMPLETED = True
print(f'Forecast, model loading and output saving: {forecast_seconds:.1f} seconds')

In [ ]:
if not RUN_COMPLETED:
    raise RuntimeError('Complete the forecast cell before reading its results.')
runtime = json.loads((RUN / 'runtime.json').read_text())
print(f"Eight model steps only: {runtime['inference_seconds']:.2f} seconds")
if runtime['cuda_peak_allocated_gib'] is not None:
    print(f"Peak PyTorch-allocated GPU memory: {runtime['cuda_peak_allocated_gib']:.2f} GiB")
print('These measurements describe this run, not a guaranteed runtime on other hardware.')
with np.load(RUN / 'forecast.npz', allow_pickle=False) as saved:
    print('Saved variables:', ', '.join(saved['variables'].tolist()))
    print('Valid times (UTC):', np.datetime_as_string(saved['times'], unit='h').tolist())
print('Results:', RUN.relative_to(ROOT))

## 3. Compare with ERA5 and persistence

A moving weather map is not enough to judge a forecast. Compare the predictions with ERA5 and with **persistence**: keep the initial weather field unchanged at every future time.

The evaluator reports latitude-area-weighted RMSE globally and over **East Asia (10–55° N, 100–160° E)**. Smaller errors are better. Wind-vector error uses both horizontal components, so it measures differences in direction as well as speed. The initial frame is excluded from forecast scores.

Pressure errors are in hPa; temperature errors are in K (a difference of 1 K equals a difference of 1 °C). The case checks were specified before this forecast was run. They are not leaderboard points or evidence of operational forecasting accuracy.

In [ ]:
if not RUN_COMPLETED:
    raise RuntimeError('Complete the forecast cell before evaluating it.')
EVALUATION_COMPLETED = False
EVALUATION = RUN / 'evaluation'
evaluation_seconds = run_script('evaluate_weather.py',
                                '--forecast', RUN / 'forecast.npz',
                                '--truth', DATA / 'truth.npz',
                                '--output-dir', EVALUATION,
                                '--coastline', COASTLINE,
                                '--coastline-attribution', COASTLINE_CREDIT,
                                '--acceptance', ASSETS / 'weather-acceptance.json', '--gif')
summary = json.loads((EVALUATION / 'evaluation.json').read_text())
EVALUATION_COMPLETED = True
print(f'Evaluation, maps and animation: {evaluation_seconds:.1f} seconds')
print('Reference:', summary['reference_kind'])
checks = summary['acceptance']['checks']
print(f"Prespecified case checks met: {sum(check['passed'] for check in checks)}/{len(checks)}")
print('Inspect the errors below, including locations where the forecast is less accurate.')

In [ ]:
if not EVALUATION_COMPLETED:
    raise RuntimeError('Complete the evaluation cell before viewing its results.')
for region, records in summary['metrics'].items():
    rows = [
        f"### {region.replace('_', ' ').title()}", '',
        'Each entry is **forecast RMSE / persistence RMSE**.', '',
        '| Lead | Wind vector (m/s) | Pressure (hPa) | Temperature (K) |',
        '|---:|---:|---:|---:|',
    ]
    for row in records:
        wind = row['wind_10m']
        pressure = row['variables']['msl']
        temperature = row['variables']['t2m']
        rows.append(
            f"| +{row['lead_hours']} h "
            f"| {wind['vector_rmse']:.2f} / {wind['persistence_vector_rmse']:.2f} "
            f"| {pressure['display_rmse']:.2f} / {pressure['display_persistence_rmse']:.2f} "
            f"| {temperature['rmse']:.2f} / {temperature['persistence_rmse']:.2f} |"
        )
    display(Markdown('\n'.join(rows)))
display(Image(filename=str(EVALUATION / 'rmse-by-lead-global.png')))
display(Image(filename=str(EVALUATION / 'rmse-by-lead-east_asia.png')))

## 4. Inspect the weather maps

Change `variable` and `lead_hours`, then rerun the next cell. **Start with +6, +24 and +48 hours.** This changes only the display; you do not need to repeat the forecast.

Compare the location and intensity of the strongest winds. Then inspect pressure and temperature. Matching the broad pattern does not mean the model gets every local extreme right.

Forecast and reanalysis share the same color scale, fixed across all eight leads. The error panel uses a separate symmetric scale. Wind arrows show direction and magnitude; the coastline provides geographic context.

In [ ]:
import matplotlib.pyplot as plt
from evaluate_weather import (
    PLOT_UNITS, draw_map, geographic_lines, load_pair, regional_plot_data,
)

variable = 'wind_speed'  # 'wind_speed', 'msl', 't2m', 'u10m', or 'v10m'
lead_hours = 24          # 6, 12, 18, 24, 30, 36, 42, or 48

if not EVALUATION_COMPLETED:
    raise RuntimeError('Complete the evaluation cell before viewing its maps.')
if variable not in PLOT_UNITS or lead_hours not in range(6, 49, 6):
    raise ValueError('Choose a listed variable and a six-hour lead from 6 to 48.')
data = load_pair(RUN / 'forecast.npz', DATA / 'truth.npz')
plot = regional_plot_data(data, variable)
frame = data['lead_hours'].tolist().index(lead_hours)
lines = geographic_lines(COASTLINE)
field_names = {'wind_speed': '10 m wind speed', 'msl': 'Mean sea-level pressure',
               't2m': '2 m temperature', 'u10m': '10 m eastward wind',
               'v10m': '10 m northward wind'}

with plt.rc_context({'font.size': 12, 'axes.titlesize': 14}):
    fig, axes = plt.subplots(1, 3, figsize=(16, 5), constrained_layout=True)
    for ax, key, title in zip(axes, ('reference', 'prediction', 'error'),
                              ('ERA5 reanalysis', 'FourCastNet forecast', 'Forecast minus ERA5')):
        is_error = key == 'error'
        limits = (-plot['error_limit'], plot['error_limit']) if is_error else plot['limits']
        vector = None
        if variable == 'wind_speed':
            vector = (plot['prediction_uv'][frame] - plot['reference_uv'][frame]
                      if is_error else plot[key + '_uv'][frame])
        image = draw_map(ax, plot[key][frame], plot['latitude'], plot['longitude'],
                         limits, 'RdBu_r' if is_error else plot['cmap'], lines, vector)
        ax.set_title(title)
        fig.colorbar(image, ax=ax, orientation='horizontal',
                     label=PLOT_UNITS[variable] + (' difference' if is_error else ''))
    valid_time = np.datetime_as_string(data['times'][frame], unit='h')
    fig.suptitle(f'{field_names[variable]} | +{lead_hours} h | {valid_time} UTC', fontsize=16)
    plt.show()
print(COASTLINE_CREDIT)
del data, plot

### Watch the 48-hour evolution

The animation shows **ERA5 reanalysis**, **FourCastNet forecast** and **forecast error** side by side. Follow the moving wind patterns, then look for differences in their position and strength. The displayed time is UTC.

In [ ]:
if not EVALUATION_COMPLETED:
    raise RuntimeError('Complete the evaluation cell to create the animation.')
display(Image(filename=str(EVALUATION / 'east-asia-wind-6-to-48h.gif')))

## Discuss the results

1. Compare +6, +24 and +48 hours. Which feature moves or changes? Does the forecast place it where ERA5 does?
2. Does FourCastNet beat persistence for wind, pressure and temperature? Use the error table, not only the maps.
3. Can a small regional RMSE hide a large error at a storm center? Point to an example or explain what you would check.
4. Why can errors grow when a model repeatedly receives its own output? Must every variable's error increase at every step?
5. How does using this pretrained model differ from training the PINNs in Labs 1–3?

**Limits of this experiment:** this is one historical initialization, not a live forecast service. Eight leads from the same initialization are not eight independent cases. Reanalysis is not direct observation. Testing broader forecast skill requires more dates and weather regimes.

This weather lab replaces the former Lab 4 in the main course. The [previous Navier–Stokes PINN notebook](../../ETC/reference_labs/04_navier_stokes/Lab_4_Navier_Stokes.ipynb) is preserved as reference material, not as a validated weather forecast.

## Sources

- [NVIDIA FourCastNet checkpoint](https://huggingface.co/nvidia/fourcastnet1): pretrained 26-channel AFNO, Apache-2.0. Revision `c67a63995f6c8e0e557eb3d791f32f437e9b02d5`.
- [Earth2Studio FCN implementation](https://github.com/NVIDIA/earth2studio/blob/486c5daa98841b0ce93cb78cb70d0dee7a8a56e6/earth2studio/models/px/fcn.py): variable order, grid and normalization.
- [ARCO-ERA5](https://github.com/google-research/arco-era5): ERA5 from the Copernicus Climate Change Service at ECMWF, distributed through Google's public archive. The local `provenance.json` records the data sources and preprocessing.
- [Natural Earth](https://www.naturalearthdata.com/about/terms-of-use/): public-domain coastline.

Maps display pressure in hPa and temperature in °C. The helper scripts preserve the model's predicted values; they do not adjust predictions to match the reference.

[Start Here](../../Start_Here.ipynb) · Previous: [Lab 3: Heat Conduction](../03_heat_conduction/Lab_3_Heat_Conduction.ipynb) · Next: [Challenge 1: Wave Dynamics](../../02_challenges/01_wave/Challenge_1_Wave_Dynamics.ipynb)